# Model registry: which digit classifier should we promote?

## Learning objectives

- Register model versions and assign a stage with `gto`, which keeps the registry in Git tags.
- Choose a model on validation accuracy, and keep the test rows for the final score.
- Write a test that catches overlapping data splits.
- Push every registered version, so an empty folder can restore any of them.

## The problem

- `project` serves one model, a small neural network that reads handwritten digits. Version v1.0.0 is in prod.
- The branch `candidate-v2` reports a validation accuracy of 1.000 and asks for promotion.
- The first release, v1.0.0, can't be restored from a clean clone.

## What you'll do

1. Step 1: train another width, register it, and decide which version goes to prod. This is the short path, about 15 minutes, and it has its own check.
2. Step 2, optional: test the candidate's data splits, and refuse the promotion on the record.
3. Step 3, optional: make every registered version restorable from an empty folder.

- Steps 2 and 3 take about 25 more minutes.
- Only reading? Each step shows the failure and how to approach the fix.

## The rule

- A model registry answers two questions: which versions exist, and which one is live.
- `gto` keeps the registry in Git tags, so there's no server to run. DVC keeps the model files.
- The model file is cattle, because you can rebuild it. The record of what you released and why is the pet.

## Get a terminal

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-04-registry?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.
- The terminal is on the left, already in the `project` folder. This page is on the right.
- Copy a command: select it and press Ctrl+C, or Cmd+C on a Mac.
- Paste it in the terminal: Ctrl+Shift+V, or right-click and choose Paste. Cmd+V on a Mac is untested, so right-click if it doesn't work.
- Edit a file: double-click it in the file panel on the far left. Save with Ctrl+S.
- A session is temporary and has 2 GB of memory. Download anything you want to keep.

**Or use your own terminal**

- Linux or macOS, on x86_64 or ARM, or Windows through WSL2. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.
- You need [conda](https://github.com/conda-forge/miniforge). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-04-registry --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git registry
cd registry
conda env create --quiet --file .binder/environment.yml
conda activate repro-registry
python scripts/setup_project.py
cd project
```

- Every command below runs in that `project` folder.
- To start over, go to the folder above it and run `python scripts/setup_project.py --reset`. This deletes your work.

## The first release

- `project` has a pipeline, a first release in prod, and the branch `candidate-v2`, which asks for promotion.
- `../osn-bucket` stands in for an object storage bucket such as OSN.
- `params.yaml` sets the width of the network, and `dvc repro` trains it.
- `split.py` writes the training, validation and test rows to `splits.json`, and Git keeps that file.
- `train.py` reports accuracy on the validation rows only.
- `score.py` asks the registry what's in prod, gets that version and scores it on the test rows. It's the only place the test rows get used.

**The commands that made the first release.** You don't need to run these.

- `gto register` gives a commit a version.
- `gto assign` says which version is in which stage.
- Both only make Git tags.

```text
dvc repro
git add .
git commit -m "Train with 32 hidden units"
gto register digits-clf --version v1.0.0
gto assign digits-clf --version v1.0.0 --stage prod
```

Look at the registry:

```bash
cat params.yaml
gto show
python score.py
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("cat params.yaml", "gto show", "python score.py")

$ cat params.yaml
split:
  seed: 2026
train:
  # Hidden units in the network's one hidden layer.
  hidden: 32

$ gto show
╒════════════╤══════════╤═════════╕
│ name       │ latest   │ #prod   │
╞════════════╪══════════╪═════════╡
│ digits-clf │ v1.0.0   │ v1.0.0  │
╘════════════╧══════════╧═════════╛

$ python score.py
prod is v1.0.0. Accuracy on the 360 test rows: 0.978


## Step 1: train another width, and decide

**Facts**

- Bigger isn't always better. Pick a width yourself.
- Decide on validation accuracy only, so compare `val_accuracy`.
- If you choose with the test rows, the test score stops being an honest estimate of how the model does on digits it has never seen.

**Steps**

1. Set `hidden` in `params.yaml` to your width, and save it. Then train and compare.

```bash
dvc repro
dvc metrics diff digits-clf@v1.0.0
```

2. Commit it, and register it as a new version.

```bash
git add .
git commit --quiet -m "Train with a new width"
gto register digits-clf --version v1.1.0
```

3. Decide. If your new width won, promote it with this. If it didn't, skip this line, because prod is already right.

```bash
gto assign digits-clf --version v1.1.0 --stage prod
```

4. Push, and look at the record.

```bash
dvc push
gto show
gto history digits-clf
python score.py
```

### Check the short path

- Run this when step 1 is done. It finds the registered version with the best validation accuracy, then clones your project into an empty folder and rebuilds prod from scratch.
- Each line says what to do next.
- When it passes, prod is your best honest model, and anyone can serve it from the remote. That's the end of the short path.

```bash
python ../check.py --short
```

On the starter project it fails like this:

In [2]:
# Recorded output. You don't need to run this.
run("python ../check.py --short")

$ python ../check.py --short
FAIL  Register at least two versions. Train a second width, commit it, and use gto register (step 1).
PASS  Prod is the version with the best validation accuracy
FAIL  A fresh clone can't serve prod, which is v1.0.0. Its model isn't in the bucket. In project/, run git checkout digits-clf@v1.0.0, then dvc push, then git checkout main.

1 of 3 parts pass.


## Step 2 (optional): a promotion to refuse

**Problem:** `candidate-v2` asks for promotion. Here's the request, and how it compares with the first release.

```bash
git show candidate-v2:PROMOTION_REQUEST.md
dvc metrics diff digits-clf@v1.0.0 candidate-v2
```

In [3]:
# Recorded output. You don't need to run this.
run("git show candidate-v2:PROMOTION_REQUEST.md", "dvc metrics diff digits-clf@v1.0.0 candidate-v2")

$ git show candidate-v2:PROMOTION_REQUEST.md
# Please promote candidate-v2

Validation accuracy is 1.000, up from what we have in prod. I also gave the
training step more data, which always helps. Can you register this and move it
to prod before the review?

Sam

$ dvc metrics diff digits-clf@v1.0.0 candidate-v2
Path          Metric        digits-clf@v1.0.0    candidate-v2    Change
metrics.json  val_accuracy  0.978                1.0             0.022


**Facts**

- A digit classifier that never gets a digit wrong is suspicious. A perfect score can mean the model has seen the answers.
- The pipeline ran with no errors, and the score is real, so no tool flags this. You need a test.
- `splits.json` holds three lists of row numbers, under `train`, `val` and `test`. `candidate-v2` has its own copy.

**Steps**

1. Write `tests/test_split.py`. It reads `splits.json` from the project folder, and fails if any row is in more than one list.
2. Run it on `main`, where it should pass, and on `candidate-v2`, where it should fail. The test file isn't committed yet, so it stays in your folder when you switch branches.

```bash
python -m pytest -q tests/test_split.py
git checkout --quiet candidate-v2
python -m pytest -q tests/test_split.py
git checkout --quiet main
```

3. Write `decisions.md`. Refuse the promotion, and say what's wrong.
4. Commit both.

```bash
git add tests decisions.md
git commit --quiet -m "Test that the splits don't overlap, and refuse candidate-v2"
```

## Step 3 (optional): get an old release back

**Problem**

- A year from now, someone needs to rerun the model behind an old result.
- `stranger.py` clones your project into an empty folder, with an empty cache, and tries to restore every registered version by its tag.

```bash
python ../scripts/stranger.py
```

In [4]:
# Recorded output. You don't need to run this.
run("python ../scripts/stranger.py")

$ python ../scripts/stranger.py
digits-clf@v1.0.0: FAILED. Its model isn't in the bucket, so a stranger can't get it.


**Facts**

- The tag and the pointer are there. The model isn't in the bucket, because nobody ran `dvc push` for that release.
- A plain `dvc push` only sends what your current files point to, so a later push doesn't save an earlier version.
- The model is only in this folder's cache. When the folder goes, the release goes with it.

**Steps**

1. Get every registered version into the bucket. `dvc push --help` shows more than one way.

```bash
dvc push --help
```

2. Run `stranger.py` again until every version restores.

## Check everything

- The full check, for steps 1 to 3.
- It runs your split test on `candidate-v2`'s splits and on some bad splits of its own, and it restores each version into an empty folder.
- It works in throwaway clones, so your project is safe.

```bash
python ../check.py
```

On the starter project it fails like this:

In [5]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
FAIL  Register at least two versions. Train a second width, commit it, and use gto register (step 1).
PASS  Prod is the version with the best validation accuracy
FAIL  A fresh clone can't serve prod, which is v1.0.0. Its model isn't in the bucket. In project/, run git checkout digits-clf@v1.0.0, then dvc push, then git checkout main.
FAIL  Commit tests/test_split.py, which reads splits.json and tests that the row lists don't overlap (step 2).
FAIL  Commit decisions.md. Name candidate-v2 and say what's wrong with its rows, so the next person doesn't have to find it again.
FAIL  These versions aren't in the bucket: v1.0.0. See step 3.

1 of 6 parts pass.


## Summary

- A registry records which versions exist and which one is live. `gto` keeps both in Git tags.
- Choose on validation. Score once, on test.
- A perfect score is a reason to test, not to promote.
- A release you can't restore from an empty folder isn't a release. Push every version.
- Write down why you promoted or refused, in the project.

## On your own project

- Get the last model you shared back from a clean machine, knowing only its version. If you can't, register it and push it.
- Write down why it was chosen over the others.
- Write the one test that would have caught your worst mistake.

## About the data

- The Optical Recognition of Handwritten Digits data: 1,797 images of 8 by 8 pixels.
- It's included in scikit-learn as `load_digits`, so there's nothing to download.
- The UCI Machine Learning Repository publishes it under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). scikit-learn's code is BSD licensed.
- Cite it as: Alpaydin, E. and Kaynak, C. (1998). Optical Recognition of Handwritten Digits. UCI Machine Learning Repository. doi:10.24432/C50P49.